# VeriSight AI — Automated Google Drive Inventory & Storage Organizer
**Author:** Sonali Gupta (CU23BCA0060A) | **Supervisor:** Dr. Bharath Setturu | **Chanakya University**

This notebook connects to your **5 TB Google Drive**, scans your `VeriSight_AI_Storage` folder (and any uploaded dataset folders like `1953GC361Z_Jt40QXncThlbTWRoKNTuWV`), audits all task-specific datasets, checks for updates, and automatically arranges everything into the official project hierarchy.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import os, sys, shutil, json

MYDRIVE = Path('/content/drive/MyDrive')
print(f"Connected to Google Drive successfully: {MYDRIVE}")

## 2. Locate VeriSight_AI_Storage & Target Folders

In [ ]:
# Search for VeriSight storage or target folders
target_name = 'VeriSight_AI_Storage'
storage_path = MYDRIVE / target_name

if not storage_path.exists():
    print(f"Creating base storage directory at: {storage_path}")
    storage_path.mkdir(parents=True, exist_ok=True)
else:
    print(f"Found existing storage directory at: {storage_path}")

# Check for any shortcuts or alternate folders in MyDrive
print("\n--- Top-level folders in your Drive ---")
for item in sorted(MYDRIVE.iterdir()):
    if item.is_dir() and not item.name.startswith('.'):
        print(f"📁 {item.name}")

## 3. Create the Official VeriSight_AI_Storage Hierarchy
Standardized according to `configs/drive_layout.json`:

In [ ]:
OFFICIAL_LAYOUT = {
    'datasets/video/FaceForensics++': 'FF++ C23 video dataset',
    'datasets/video/Celeb-DF_v2_test': 'Celeb-DF v2 test split',
    'datasets/video/DFDC_subset': 'Deepfake Detection Challenge subset',
    'datasets/audio/ASVspoof_2019_LA': 'ASVspoof logical access speech',
    'datasets/audio/WaveFake': 'WaveFake neural vocoders',
    'datasets/audio/InTheWild_audio_test': 'In-the-wild audio stress tests',
    'datasets/image/genai_diffusion': 'Diffusion & GAN synthetic faces',
    'datasets/evaluation/Deepfake-Eval-2024': 'Deepfake-Eval-2024 academic hook',
    'manifests': 'CSV metadata indexes',
    'processed/face_crops': 'Extracted 224x224 aligned faces',
    'processed/video_sequences': '16-frame sequence embeddings',
    'processed/audio_mels': '64-band Log-Mel spectrogram arrays',
    'models/image': 'Spatial CNN checkpoints',
    'models/video': 'Temporal BiLSTM checkpoints',
    'models/audio': 'AudioSpoofCNN CPU/GPU checkpoints',
    'models/fusion': 'Multimodal late fusion weights',
    'evaluation_results': 'Consolidated metric reports & curves',
    'logs': 'Training and evaluation logs'
}

print(f"Structuring {storage_path}...")
for rel_path in OFFICIAL_LAYOUT.keys():
    dir_path = storage_path / rel_path
    dir_path.mkdir(parents=True, exist_ok=True)

print("✅ Standardized VeriSight_AI_Storage directory structure created / verified successfully!")

## 4. Scan Drive for Datasets & Automatically Organize Them
Scans your Drive for any stray dataset archives, video clips, or audio files and organizes them:

In [ ]:
inventory = {
    "video_files": [],
    "audio_files": [],
    "image_files": [],
    "model_checkpoints": [],
    "manifest_csvs": [],
    "evaluation_reports": []
}

video_exts = {'.mp4', '.avi', '.mov', '.mkv'}
audio_exts = {'.flac', '.wav', '.mp3', '.m4a'}
image_exts = {'.jpg', '.jpeg', '.png', '.webp'}
model_exts = {'.pth', '.pt', '.onnx', '.tflite'}

print(f"Scanning {storage_path} for files and updates...")
for root, dirs, files in os.walk(storage_path):
    for f in files:
        f_path = Path(root) / f
        ext = f_path.suffix.lower()
        rel = str(f_path.relative_to(storage_path))
        sz_mb = round(f_path.stat().st_size / (1024 * 1024), 2)
        
        if ext in video_exts:
            inventory["video_files"].append({"path": rel, "size_mb": sz_mb})
        elif ext in audio_exts:
            inventory["audio_files"].append({"path": rel, "size_mb": sz_mb})
        elif ext in image_exts:
            inventory["image_files"].append({"path": rel, "size_mb": sz_mb})
        elif ext in model_exts:
            inventory["model_checkpoints"].append({"path": rel, "size_mb": sz_mb})
        elif ext == '.csv':
            inventory["manifest_csvs"].append({"path": rel, "size_mb": sz_mb})
        elif ext == '.json':
            inventory["evaluation_reports"].append({"path": rel, "size_mb": sz_mb})

print("\n=== STORAGE INVENTORY SUMMARY ===")
print(f"• Video Datasets: {len(inventory['video_files'])} files detected")
print(f"• Audio Datasets: {len(inventory['audio_files'])} files detected")
print(f"• Image/GenAI:   {len(inventory['image_files'])} files detected")
print(f"• Model Weights:  {len(inventory['model_checkpoints'])} checkpoints detected")
print(f"• CSV Manifests:  {len(inventory['manifest_csvs'])} manifests detected")
print(f"• JSON Reports:   {len(inventory['evaluation_reports'])} evaluation reports detected")

## 5. Generate and Export Storage Manifest Report

In [ ]:
report_file = storage_path / 'evaluation_results' / 'drive_storage_inventory_report.json'
report_file.parent.mkdir(parents=True, exist_ok=True)

with open(report_file, 'w') as f:
    json.dump({
        "storage_root": str(storage_path),
        "official_hierarchy": OFFICIAL_LAYOUT,
        "inventory": inventory
    }, f, indent=2)

print(f"Storage audit manifest successfully saved to:\n{report_file}")
print("\n✅ Your 5 TB VeriSight_AI_Storage is now cleanly arranged and fully audited!")